# 14 · Convert corpus80 shard 2 into one compressed file per study (Kaggle, CPU)

**Why?** We wanted to train Raptor-80 on a local RTX 5090 server with only ~36 GB of free disk; the raw corpus is
40 GB. This notebook writes every study of the shard to its own compressed `.npz` (file name = row number in the
joined corpus). The server downloads only these compressed files and the training script reads them through
`RAPTOR_NPZ_DIR` (`third_party/raptor/train_knee.py`).

In [ ]:
import os, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np

SHARD = 2
def find(name):
    for d, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
        if name in files:
            return Path(d) / name
    raise FileNotFoundError(name)

vols = np.load(find(f"corpus_vols_s{SHARD:02d}.npy"), mmap_mode="r")
masks = np.load(find(f"corpus_masks_s{SHARD:02d}.npy"))
ids = np.load(find(f"corpus_ids_s{SHARD:02d}.npy"), allow_pickle=True)
OFFSET = SHARD * 1469            # shards of 1,469 studies (4,407 / 3); row = OFFSET + i
OUT = Path("/kaggle/working/npz"); OUT.mkdir(parents=True, exist_ok=True)
print(vols.shape, OFFSET)

t0 = time.time()
def save(i):
    np.savez_compressed(OUT / f"{OFFSET + i:05d}.npz", v=np.asarray(vols[i]))
with ThreadPoolExecutor(8) as ex:
    list(ex.map(save, range(len(vols))))
np.save(f"/kaggle/working/masks_s{SHARD:02d}.npy", masks)
np.save(f"/kaggle/working/ids_s{SHARD:02d}.npy", ids, allow_pickle=True)
size = sum(f.stat().st_size for f in OUT.glob("*.npz")) / 1e9
print(f"{len(vols)} studies -> {size:.1f} GB compressed ({size / (vols.nbytes / 1e9):.0%}), {(time.time() - t0) / 60:.1f} min")